# 04. Casos Curados de Tesis (`thesis_curated`) y Cálculo de Ampacidad

Trabaja con los casos de cables subterráneos multicapa (`xlpe_discrete_layers`, `xlpe_backfill`) e
ilustra la inicialización sin malla `rbf_readout`, el muestreo focalizado y la bisección térmica
de ampacidad mediante `AmpacitySolver`.

In [2]:
import sys
from pathlib import Path

# Localiza PINN_V4 automáticamente aunque se ejecute desde dentro de notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Raíz PINN_V4:", ROOT)

Raíz PINN_V4: C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_2026-2_pinn


In [3]:
import pinn_thermal.models
from pinn_thermal.core.ampacity import AmpacitySolver
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.visualization import GridSampler
from frontend.state import WorkspaceStore

store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
prob = store.load_problem("xlpe_discrete_layers")
print("Regiones de xlpe_discrete_layers:", [(r.name, r.kind, r.priority) for r in prob.regions])

cfg = TrainConfig(
    adam_steps=200,
    lbfgs_steps=15,
    width=24,
    depth=3,
    focused=True,
    resample_every=100,
    initializer="rbf_readout",
)
art = PINNTrainer(cfg).train(prob, "M1_EXACT", store.runs_dir / "nb04_xlpe_M1_EXACT", overwrite=True)
print("Evaluación M1_EXACT:", art.result["evaluation"])

# Corte vertical por x = 0 atravesando el cable y la interfaz de estratos y = -1.0 m
cut = GridSampler.sample_line_cut({"M1_EXACT": art}, prob, (0.0, -3.5), (0.0, 0.0), n_points=120)
print("T max sobre corte vertical:", float(max(cut["series"]["M1_EXACT"])), "°C")

# Cálculo rápido de ampacidad usando inicialización RBF + L-BFGS determinista
amp_cfg = TrainConfig(
    adam_steps=0,
    lbfgs_steps=5,
    lbfgs_iterations=5,
    width=20,
    depth=2,
    device="cpu",
    initializer="rbf_readout",
    initializer_settings={"n_per_region": 120, "stencil_size": 16},
)
amp_res = AmpacitySolver(
    R20_ohm_m=2.0e-4,
    target_temp_C=90.0,
    relaxation=0.8,
    max_iters=2,
    coupling_tol_K=1.0,
    source_tol_rel=0.05,
    max_coupling_iters=25,
).solve_with_pinn(
    prob, architecture="M1", config=amp_cfg, work_dir=store.runs_dir / "nb04_ampacity_tmp", initial_upper_A=700.0
)
print(
    f"Corriente I_mid={amp_res.I_mid_A:.2f} A | potencia={amp_res.power_W_m:.2f} W/m | "
    f"Tmax={amp_res.Tmax_at_mid_C:.2f} °C | convergió={amp_res.converged}"
)

Regiones de xlpe_discrete_layers: [('soil_lower', 'soil', 0), ('soil_upper', 'soil', 1), ('cable_0_layer_0', 'conductor', 2), ('cable_0_layer_1', 'conductor', 3), ('cable_0_layer_2', 'conductor', 4), ('cable_0_layer_3', 'conductor', 5)]


C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_2026-2_pinn\src\pinn_thermal\core\transfer.py:413: UserWarning: Aten Op fallback from XPU to CPU happends. This may have performance implications. If need debug the fallback ops please set environment variable `PYTORCH_DEBUG_XPU_FALLBACK=1`  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\xpu\ATen\RegisterXPU_0.cpp:54666.)
  theta = torch.linalg.lstsq(design, target).solution.reshape(-1)
Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Evaluación M1_EXACT: {'pde_rms': {'soil_lower': 0.31754323840141296, 'soil_upper': 0.503078281879425, 'cable_0_layer_0': 0.4407981038093567, 'cable_0_layer_1': 1.3193988124271527e-08, 'cable_0_layer_2': 3.950900051563622e-08, 'cable_0_layer_3': 3.3174213598385904e-08}, 'constitutive_rms': {}, 'interfaces': [{'name': 'soil_lower->soil_upper@1', 'T_jump_rms_K': 0.9512363076210022, 'T_jump_max_K': 1.7516975402832031, 'flux_jump_rms_pct': 14.178535461425781, 'flux_jump_max_pct': 29.16646957397461, 'derived_flux_jump_rms_pct': 14.178535461425781, 'flux_scale_W_m2': 17.54763426263378}, {'name': 'cable_0_layer_0->cable_0_layer_1@0', 'T_jump_rms_K': 0.9611839056015015, 'T_jump_max_K': 2.3530540466308594, 'flux_jump_rms_pct': 0.16356110572814941, 'flux_jump_max_pct': 0.21348024904727936, 'derived_flux_jump_rms_pct': 0.16356110572814941, 'flux_scale_W_m2': 14300000.0}, {'name': 'cable_0_layer_1->cable_0_layer_2@0', 'T_jump_rms_K': 0.5577484965324402, 'T_jump_max_K': 1.3743171691894531, 'flux_jum